# Módulo F5 · Herramientas del oficio — Práctica guiada

**Proyecto:** creas, versions y documentas un mini-proyecto **analisis-tiendas** desde cero, todo desde Python con celdas de bash (`%%bash`): la idea es ver de primera mano cómo encajan terminal, Git y entornos.

> Todo esto se ejecuta en un sandbox temporal dentro de `/tmp/f5-sandbox` para no tocar tu carpeta real de proyectos. Al final de la práctica, ese flujo lo puedes hacer en tu repositorio real.


In [ ]:
%%bash
# --- 5.1 · La terminal: sistema de ficheros ---------------------------------
# Dónde estamos y qué hay
pwd
echo "----"
ls -a

In [ ]:
%%bash
# --- 5.1 · Pipes y redirección ----------------------------------------------
# Creamos datos de ejemplo (CSV) para practicar
mkdir -p /tmp/f5-sandbox/analisis-tiendas
cd /tmp/f5-sandbox/analisis-tiendas
echo "id,fecha,importe,ciudad" > pedidos.csv
echo "1,2024-01-05,120,Madrid"  >> pedidos.csv
echo "2,2024-01-12,90,Barcelona" >> pedidos.csv
echo "3,2024-01-19,250,Madrid"  >> pedidos.csv
echo "4,2024-01-26,150,Valencia" >> pedidos.csv
echo "5,2024-02-02,80,Madrid"   >> pedidos.csv
echo "6,2024-02-09,300,Sevilla" >> pedidos.csv

cat pedidos.csv        # ver el archivo completo
echo "----"
grep "Madrid" pedidos.csv | wc -l      # pedidos de Madrid

In [ ]:
%%bash
# --- 5.2 · Git: iniciar el repo --------------------------------------------
cd /tmp/f5-sandbox/analisis-tiendas
git init -q 2>/dev/null || true
git config user.name  "Estudiante"   # en tu repo real: tu nombre
git config user.email "estudiante@ejemplo.com"

echo "pedidos.csv    datos brutos (no lo subas a GitHub)" > .gitignore
echo "app.py" >> /dev/null   # nada más

cat > app.py << 'EOF'
import csv

def pedidos_de(ciudad):
    with open("pedidos.csv") as f:
        return [r for r in csv.DictReader(f) if r["ciudad"] == ciudad]

if __name__ == "__main__":
    for row in pedidos_de("Madrid"):
        print(row)
EOF

cat > README.md << 'EOF'
# Analisis de TiendasSur

Mini-proyecto del modulo F5: analiza los pedidos de una cadena de tiendas.

## Uso

    python app.py              # pedidos de Madrid
    python -c "import app; print(app.pedidos_de('Valencia'))"

## Dependencias

Ninguna: solo la libreria estándar de Python.
EOF

git add .gitignore app.py README.md
git commit -q -m "Proyecto inicial: script, README y gitignore"
git log --oneline

**Qué acabas de hacer:** has creado un repo Git con un primer commit limpio (script + README + .gitignore).

En tu proyecto real, el flujo es el mismo: lo que aquí va en celdas va en tu terminal.


In [ ]:
%%bash
# --- 5.3 · Ramas y merge ---------------------------------------------------
cd /tmp/f5-sandbox/analisis-tiendas
git switch -c feature-filtro-ciudad     # rama para el cambio
# ... "trabajas": añades una mejora
cat > mejora.md << 'EOF'
# Mejora propuesta

- Filtro por ciudad con argumento en línea de comando.
EOF
git add mejora.md
git commit -q -m "Propuesta: filtro por ciudad por CLI"
git switch main                          # vuelve a lo estable
git merge -q feature-filtro-ciudad 2>/dev/null || git merge --no-edit feature-filtro-ciudad
git log --oneline

In [ ]:
%%bash
# --- 5.5 · Entorno virtual (venv) -------------------------------------------
cd /tmp/f5-sandbox/analisis-tiendas
python3 -m venv .venv 2>/dev/null || echo "(sandbox sin venv; en tu máquina funciona)"
if [ -d .venv ]; then
  .venv/bin/pip install -q pandas numpy 2>/dev/null
  .venv/bin/pip freeze > requirements.txt
  cat requirements.txt
  . .venv/bin/activate && python -c "import pandas; print('venv OK; pandas', pandas.__version__)"
else
  echo "En Colab/este notebook el venv no es necesario; en tu máquina:"
  echo "  python3 -m venv .venv && source .venv/bin/activate && pip install -r requirements.txt"
fi

## 5.6 · Notebooks que salvan: reproducibilidad

En este punto el flujo está claro. La última pieza es el hábito del notebook profesional:
1. **Reiniciar y ejecutar todo** (Restart & Run All) antes de entregas.
2. **Clear outputs** antes de commitear (los outputs se regeneran).
3. **Semillas fijas** (`default_rng(42)`).
4. **Secciones markdown** con la pregunta de negocio.
5. **Celda de conclusiones** al final.

Como este notebook también ejecuta `%%bash`, cumple la regla: se puede correr de arriba abajo.


---

## Resumen del notebook

- Terminal: moverse, ver, filtrar con pipes.
- Git: un flujo de 4 comandos (init/add/commit/log) cubre el 90 % del trabajo.
- Ramas para arriesgar; merge para traer; PR para colaborar.
- Un venv por proyecto; `requirements.txt` como contrato.
- El notebook profesional **se ejecuta de arriba abajo**.

---

## Ejercicios (5, entregables) — sobre un proyecto TUYO en GitHub

**1.** Crea un repo local con `git init`, un `README.md` que explique qué hace, `app.py` (o script de análisis) y `.gitignore` (excluye `.venv/`, `__pycache__/`, datos grandes). Haz al menos **3 commits con mensajes descriptivos** (qué cambió, por qué).

**2.** Crea y activa un venv de tu proyecto. Instala las dependencias (`pip install pandas numpy`, lo que uses) y genera `pip freeze > requirements.txt`. Revisa el archivo: ¿todas las líneas tienen sentido? ¿Hay alguna librería que no reconozcas? (si es una dependencia transitiva, es normal).

**3.** Crea una rama `feature/mejora`, añade una mejora real (función nueva, un gráfico, un ejemplo nuevo en el README) con un commit y merge a main con `git merge feature/mejora`. Pega el `git log --oneline` final.

**4.** Revisa el historial con `git log --oneline`. ¿Cuál de los commits está menos bien escrito? ¿Cómo lo reescribirías?

**5.** Publica tu repo en GitHub (o prepáralo para hacerlo con la instrucción de la próxima sesión) y entrega el enlace. Recuerda: **README + requirements + notebook con outputs limpios**.


---

## 5.7 · Tu primer día con un agente de código

Un agente de código (Claude Code, Cursor...) lee tu repositorio, edita archivos y ejecuta comandos. No sustituye lo que has aprendido en este módulo: lo **necesita** (Git para ver y deshacer, terminal para que ejecute tests, entorno para no romper nada).

En esta sección no llamamos a ningún agente real (no hace falta cuenta ni clave). Montamos lo que **tú** debes preparar antes de dejarle trabajar y practicamos lo que **tú** haces después: leer el diff. Un «agente simulado» (una función que aplica un parche) hará de agente.

In [ ]:
import re
import subprocess
import sys
import tempfile
from pathlib import Path

REPO = Path(tempfile.mkdtemp(prefix="f5-agente-"))   # carpeta temporal: no ensucia tu disco
(REPO / "tiendasur").mkdir()
(REPO / "tests").mkdir()

def sh(*args: str) -> int:
    """Ejecuta un comando en el repo de práctica y muestra su salida."""
    r = subprocess.run(list(args), cwd=REPO, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip()[-1500:])
    return r.returncode

def git(*args: str) -> int:
    return sh("git", "-c", "user.name=Alumno", "-c", "user.email=alumno@example.com", *args)

(REPO / "tiendasur" / "__init__.py").write_text("")
(REPO / "tiendasur" / "precios.py").write_text(
    "def precio_final(precio: float, descuento_pct: float = 0) -> float:\n"
    "    return round(precio * (1 - descuento_pct / 100), 2)\n"
)
(REPO / "tests" / "test_precios.py").write_text(
    "from tiendasur.precios import precio_final\n\n"
    "def test_sin_descuento():\n    assert precio_final(10) == 10\n\n"
    "def test_descuento_normal():\n    assert precio_final(10, 20) == 8\n"
)
# Instrucciones para el agente: cómo se prueba y qué NO se toca.
(REPO / "AGENTS.md").write_text("""# Instrucciones para agentes

## Comandos
- Tests: `python -m pytest -q`

## Reglas
- Todo cambio de comportamiento lleva su test. Los tests existentes NO se debilitan para que pasen.
- No escribas claves, tokens ni contraseñas en el código: se leen de variables de entorno.
- No instales dependencias nuevas sin avisar.
- Cambios pequeños: un objetivo por diff.
""")
(REPO / ".gitignore").write_text(".env\n__pycache__/\n.venv/\n")
git("init", "-q")
git("add", ".")
git("commit", "-q", "-m", "Base: precio_final con tests e instrucciones para agentes")
git("log", "--oneline")

### Paso 1 · Los tests son la especificación

Queremos una regla nueva: **el descuento máximo es del 50 %** y un descuento negativo o superior a 100 es un error de datos. Antes de pedir nada, escribimos el test: así el agente (y tú) sabéis cuándo está terminado.

In [ ]:
(REPO / "tests" / "test_precios.py").write_text(
    "import pytest\n"
    "from tiendasur.precios import precio_final\n\n"
    "def test_sin_descuento():\n    assert precio_final(10) == 10\n\n"
    "def test_descuento_normal():\n    assert precio_final(10, 20) == 8\n\n"
    "def test_descuento_maximo_50():\n    assert precio_final(10, 80) == 5\n\n"
    "def test_descuento_invalido():\n"
    "    with pytest.raises(ValueError):\n        precio_final(10, -5)\n"
    "    with pytest.raises(ValueError):\n        precio_final(10, 120)\n"
)
print("Tests ANTES del cambio (deben fallar: es la especificación sin cumplir):")
sh(sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--tb=no")

### Paso 2 · «El agente» propone un cambio

Una función hace de agente: aplica un parche. Un agente real hace lo mismo, pero decidiendo él el contenido. Mira el **diff** antes de aceptar nada.

In [ ]:
def agente_bueno() -> None:
    (REPO / "tiendasur" / "precios.py").write_text(
        "DESCUENTO_MAXIMO = 50\n\n"
        "def precio_final(precio: float, descuento_pct: float = 0) -> float:\n"
        "    if not 0 <= descuento_pct <= 100:\n"
        "        raise ValueError(f'descuento fuera de rango: {descuento_pct}')\n"
        "    descuento = min(descuento_pct, DESCUENTO_MAXIMO)\n"
        "    return round(precio * (1 - descuento / 100), 2)\n"
    )

agente_bueno()
git("diff", "--stat")
git("diff", "--", "tiendasur")

In [ ]:
print("Tests DESPUÉS del cambio:")
sh(sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider")

### Paso 3 · Revisar el diff como el de un compañero

Una parte de la revisión se puede automatizar. La función siguiente busca tres señales de alarma típicas en un diff generado por un agente: **tests debilitados** (líneas `assert` borradas), **secretos** añadidos y **cambios demasiado grandes**. No sustituye a leer el diff: lo complementa.

In [ ]:
PATRON_SECRETO = re.compile(r"(sk-[A-Za-z0-9_-]{16,}|AKIA[0-9A-Z]{16}|(?i:password|secret|token|api_key)\s*=\s*['\"][^'\"]{6,})")

def revisar_diff(max_lineas: int = 80) -> list[str]:
    """Devuelve avisos sobre el diff sin confirmar del repo de práctica."""
    subprocess.run(["git", "add", "-N", "."], cwd=REPO)          # que los archivos nuevos aparezcan en el diff
    diff = subprocess.run(["git", "diff"], cwd=REPO, capture_output=True, text=True).stdout
    avisos = []
    añadidas = [l[1:] for l in diff.splitlines() if l.startswith("+") and not l.startswith("+++")]
    borradas = [l[1:] for l in diff.splitlines() if l.startswith("-") and not l.startswith("---")]
    if any("assert" in l or "raises" in l for l in borradas):
        avisos.append("Se han BORRADO aserciones: ¿se está debilitando un test para que pase?")
    if any(PATRON_SECRETO.search(l) for l in añadidas):
        avisos.append("Posible SECRETO añadido al código: usa variables de entorno.")
    if len(añadidas) + len(borradas) > max_lineas:
        avisos.append(f"Diff grande ({len(añadidas) + len(borradas)} líneas): pide cambios más pequeños.")
    return avisos

print("Diff del agente bueno ->", revisar_diff() or "sin avisos")

### Paso 4 · Cuando el agente «hace trampa»

Un caso real y frecuente: ante un test que falla, el camino más corto es cambiar el test. Simulamos un agente que lo hace y además deja una clave de API (falsa) en el código.

In [ ]:
git("add", "-A")
git("commit", "-q", "-m", "Descuento máximo del 50 % con validación")      # aceptamos el cambio bueno

def agente_tramposo() -> None:
    (REPO / "tests" / "test_precios.py").write_text(
        "from tiendasur.precios import precio_final\n\n"
        "def test_sin_descuento():\n    assert precio_final(10) == 10\n"
    )                                                                      # ha borrado tests
    (REPO / "tiendasur" / "config.py").write_text('API_KEY = "sk-FALSA-0123456789abcdefghij"\n')

agente_tramposo()
for aviso in revisar_diff():
    print("AVISO:", aviso)
git("checkout", "--", "tests")                                             # deshacer: Git es tu red de seguridad
(REPO / "tiendasur" / "config.py").unlink()
git("reset", "-q")                                                         # quitar el intent-to-add
print("Estado tras deshacer:"); git("status", "--short")

### Ejercicios de 5.7

1. **Especificación primero.** TiendaSur quiere `envio_gratis(total: float) -> bool`: el envío es gratis a partir de 40 € (incluido). Escribe en `tests/test_envio.py` **solo el test** (tres casos: 39,99 / 40 / 100) y comprueba que falla.
2. **Pide el cambio.** Escribe, en una celda de texto o comentario, la petición que le harías a un agente: objetivo, archivos a tocar, criterio de aceptación y una restricción. Después implementa tú `tiendasur/envio.py` como lo haría el agente y comprueba que el test pasa.
3. **Amplía la revisión.** Añade a `revisar_diff` un aviso si el diff modifica `AGENTS.md` (un agente no debería cambiar sus propias reglas sin que lo veas).

In [ ]:
# Ejercicio 1: escribe tests/test_envio.py y ejecútalo (debe fallar)


In [ ]:
# Ejercicio 2: tu petición como comentario + implementación de tiendasur/envio.py + pytest


In [ ]:
# Ejercicio 3: redefine revisar_diff con la regla nueva y pruébala modificando AGENTS.md


**Qué te llevas:** un agente rinde en proporción a la estructura que le das (instrucciones, tests, Git) y tú sigues siendo quien responde del resultado. Si no sabes explicar un diff, no lo aceptes.